# Kaggle Scientific Smoke V2 — Dependency-Aware Selective Regeneration Benchmark

**SCIENTIFIC SMOKE V2 / NON-PUBLICATION**

Runs the minimal real Kaggle smoke using the KaggleQwenBackend on GPU.

- **Scientific Smoke V2**: 1 repository (todo) × 3 frozen scenarios (todo-smoke-001/002/003) × 3 arms × 1 run = 9 total runs
- **Arms**: monolithic, selective, iterative_repository_agent
- **Backend**: kaggle-qwen (Qwen2.5-Coder on Kaggle GPU)
- **OpenRouter**: NOT used for this smoke

Use only --profile scientific-smoke-v2. Do not switch to Pilot or Research.


In [ ]:
import os
import sys
import subprocess
import json as _json
from pathlib import Path

# ---- Discover Kaggle Datasets ----------------------------------------------
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/runs/scientific_smoke_v2")

KNOWN_CODE = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-code"
KNOWN_DATA = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-data"
KNOWN_MODEL = KAGGLE_INPUT / "models/qwen-lm/qwen2.5-coder/transformers/7b-instruct/1"
FALLBACK_CODE = KAGGLE_INPUT / "dependency-aware-selective-regeneration-code"
FALLBACK_DATA = KAGGLE_INPUT / "dependency-aware-selective-regeneration-data"

def discover(label, candidates, required_subdir=None):
    for p in candidates:
        if p.is_dir():
            if required_subdir is None or (p / required_subdir).is_dir():
                return p
            print(f"  [info] {p.name} exists but missing '{required_subdir}'")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if entry.is_dir() and (required_subdir is None or (entry / required_subdir).is_dir()):
                return entry
    raise FileNotFoundError(f"Cannot find {label} in {KAGGLE_INPUT}")

CODE_DIR = discover("code dataset", [KNOWN_CODE, FALLBACK_CODE], required_subdir="src")
DATA_DIR = discover("data dataset", [KNOWN_DATA, FALLBACK_DATA], required_subdir="scenarios")

src_dir = CODE_DIR / "src"
if src_dir.is_dir():
    sys.path.insert(0, str(src_dir))
else:
    raise FileNotFoundError(f"src/ not found in code dataset: {CODE_DIR}")

if KNOWN_MODEL.is_dir():
    MODEL_DIR = KNOWN_MODEL
else:
    MODEL_DIR = None

# Fail closed: a valid Qwen model (config.json + at least one weight file)
# must be discovered before any experiment is created. No warning-and-empty
# string behavior is allowed.
MODEL_WEIGHT_SUFFIXES = (".safetensors", ".bin")

def _has_weight_files(p: Path) -> bool:
    return any(
        f.is_file() and f.suffix in MODEL_WEIGHT_SUFFIXES
        for f in p.rglob("*")
    )

def _is_valid_model_dir(p: Path) -> bool:
    return p.is_dir() and (p / "config.json").is_file() and _has_weight_files(p)

def discover_model(candidates) -> Path:
    for p in candidates:
        if _is_valid_model_dir(p):
            return p
        print(f"  [info] {p.name}: not a valid Qwen model dir (config.json + weights required)")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if _is_valid_model_dir(entry):
                return entry
    raise FileNotFoundError(
        f"Cannot find a valid Qwen model under {KAGGLE_INPUT}: "
        "config.json and at least one .safetensors/.bin weight file required"
    )

MODEL_PATH = str(discover_model([MODEL_DIR] if MODEL_DIR else []).resolve())

SCRIPT_PATH = CODE_DIR / "seven_arm_benchmark.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"seven_arm_benchmark.py not found in {CODE_DIR}")

SOURCE_COMMIT = "ffa179ade389193082ee1a11af4d29e86c351e08"
DEPLOYED_BUILD_ID = "ffa179a"
HF_RESULTS_REPO_ID = "NabilDo/selective-regeneration-experiment-results"

print(f"Output dir:    {OUTPUT_DIR}")
print(f"Source commit: {SOURCE_COMMIT}")
print(f"Build ID:      {DEPLOYED_BUILD_ID}")
print(f"Model path:    {MODEL_PATH}")

# ---- R7B: live-run observability -------------------------------------------

class ScientificSmokeExecutionError(RuntimeError):
    'Raised when a benchmark invocation does not produce a valid result.'

EVIDENCE_FILES = (
    "checkpoint.json",
    "progress.json",
    "failure_records.json",
    "remote_sync.json",
    "run_records.jsonl",
    "dashboard/dashboard_summary.json",
    "dashboard/run_matrix.csv",
    "dashboard/strategy_summary.csv",
    "dashboard/failure_summary.csv",
)

def _load_smoke_evidence(output_dir):
    output_dir = Path(output_dir)
    evidence = {}
    for name in EVIDENCE_FILES:
        p = output_dir / name
        if not p.is_file():
            evidence[name] = None
            continue
        try:
            if name.endswith(".jsonl"):
                rows = [_json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
                evidence[name] = rows
            elif name.endswith(".json"):
                evidence[name] = _json.loads(p.read_text(encoding="utf-8"))
            else:
                evidence[name] = p.read_text(encoding="utf-8")
        except Exception as exc:
            evidence[name] = {"load_error": str(exc)}
    return evidence

def _gpu_diagnostics():
    try:
        import torch
        if not torch.cuda.is_available():
            return "cuda unavailable"
        name = torch.cuda.get_device_name(0)
        alloc = torch.cuda.memory_allocated(0) / 2**30
        reserved = torch.cuda.memory_reserved(0) / 2**30
        return f"{name} allocated={alloc:.2f}GiB reserved={reserved:.2f}GiB"
    except Exception as exc:
        return f"gpu diagnostics unavailable: {exc}"

def _display_smoke_dashboard(output_dir):
    output_dir = Path(output_dir)
    evidence = _load_smoke_evidence(output_dir)
    records = evidence.get("run_records.jsonl") or []
    if not isinstance(records, list):
        records = []
    cp = evidence.get("checkpoint.json") or {}
    if not isinstance(cp, dict):
        cp = {}
    dash_dir = output_dir / "dashboard"
    dash_dir.mkdir(parents=True, exist_ok=True)
    print("\n--- SMOKE DASHBOARD ---")
    print("KPI: planned=%s completed=%s status=%s" % (
        cp.get("total_planned"), cp.get("total_completed"), cp.get("completion_status")))
    model_calls = sum(int(r.get("total_workflow_model_calls", 0) or 0) for r in records)
    tokens = sum(int(r.get("total_workflow_tokens", 0) or 0) for r in records)
    print("KPI: model_calls=%s tokens=%s records=%s" % (model_calls, tokens, len(records)))
    try:
        import pandas as pd
        rows = []
        for r in records:
            rows.append({
                "run_id": r.get("run_id", ""),
                "scenario": r.get("scenario_id", ""),
                "strategy": r.get("strategy_id", r.get("strategy_name", "")),
                "status": r.get("status", ""),
                "model_calls": int(r.get("total_workflow_model_calls", 0) or 0),
                "tokens": int(r.get("total_workflow_tokens", 0) or 0),
                "duration_seconds": float(r.get("duration_seconds", 0) or 0),
            })
        df = pd.DataFrame(rows)
        if not df.empty:
            print("\nPer-run table:")
            print(df.to_string(index=False))
            print("\nScenario x Strategy matrix (status):")
            try:
                matrix = df.pivot_table(index="scenario", columns="strategy", values="status", aggfunc="first")
                print(matrix.to_string())
            except Exception as exc:
                print("(matrix unavailable: %s)" % exc)
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        if not df.empty:
            for col, name, how in (
                ("status", "status_by_strategy.png", "count"),
                ("tokens", "tokens_by_strategy.png", "sum"),
                ("model_calls", "model_calls_by_strategy.png", "sum"),
                ("duration_seconds", "duration_by_strategy.png", "sum"),
            ):
                try:
                    if how == "count":
                        ax = df.groupby(["strategy", "status"]).size().unstack(fill_value=0).plot(kind="bar", title=col)
                    else:
                        ax = df.groupby("strategy")[col].sum().plot(kind="bar", title=col)
                    fig = ax.get_figure()
                    fig.savefig(dash_dir / name, bbox_inches="tight")
                    plt.close(fig)
                    print("chart saved: %s" % (dash_dir / name))
                except Exception as exc:
                    print("(chart %s failed: %s)" % (name, exc))
        failed = [r for r in records if r.get("status") != "succeeded"]
        if failed:
            causes = {}
            for r in failed:
                k = r.get("failure_classification") or r.get("failure_stage") or "unknown"
                causes[k] = causes.get(k, 0) + 1
            print("\nFailure causes: %s" % causes)
            try:
                fig, ax = plt.subplots()
                ax.bar(list(causes.keys()), list(causes.values()))
                ax.set_title("failure_causes")
                fig.savefig(dash_dir / "failure_causes.png", bbox_inches="tight")
                plt.close(fig)
                print("chart saved: %s" % (dash_dir / "failure_causes.png"))
            except Exception as exc:
                print("(failure_causes.png failed: %s)" % exc)
    except Exception as exc:
        print("(dashboard tables/charts unavailable: %s)" % exc)

def _raise_actionable_smoke_error(output_dir):
    output_dir = Path(output_dir)
    evidence = _load_smoke_evidence(output_dir)
    print("\n" + "=" * 78)
    print("BENCHMARK FAILED - actionable diagnosis")
    print("=" * 78)
    _display_smoke_dashboard(output_dir)
    cp = evidence.get("checkpoint.json") or {}
    if not isinstance(cp, dict):
        cp = {}
    records = evidence.get("run_records.jsonl") or []
    if not isinstance(records, list):
        records = []
    sync = evidence.get("remote_sync.json") or {}
    if not isinstance(sync, dict):
        sync = {}
    failed = [r for r in records if r.get("status") != "succeeded"] or records[-1:]
    first = failed[0] if failed else {}
    lines = []
    lines.append("experiment/source/build: %s / %s / %s" % (
        cp.get("experiment_id", "?"),
        cp.get("source_commit", "?"),
        cp.get("deployed_build_id", "?"),
    ))
    lines.append("first failed run: %s" % first.get("run_id", "?"))
    lines.append("scenario: %s" % first.get("scenario_id", "?"))
    lines.append("strategy: %s" % (first.get("strategy_id") or first.get("strategy_name") or "?"))
    lines.append("stage: %s" % first.get("failure_stage", "?"))
    lines.append("classification: %s" % first.get("failure_classification", "?"))
    msgs = []
    for r in records:
        for d in (r.get("failure_details") or []):
            m = d.get("message") or d.get("error") or ""
            if m and m not in msgs:
                msgs.append(m)
    if msgs:
        lines.append("top unique messages: " + " | ".join(msgs[:5]))
    calls = sum(int(r.get("total_workflow_model_calls", 0) or 0) for r in records)
    tokens = sum(int(r.get("total_workflow_tokens", 0) or 0) for r in records)
    lines.append("model calls: %s" % calls)
    lines.append("tokens: %s" % tokens)
    sel = sum(int(r.get("selected_artifact_count", 0) or 0) for r in records)
    regen = sum(int(r.get("regenerated_artifact_count", 0) or 0) for r in records)
    lines.append("selected/regenerated: %s/%s" % (sel, regen))
    lines.append("GPU/OOM: %s" % _gpu_diagnostics())
    lines.append("HF state: %s" % sync.get("last_sync", "?"))
    lines.append("evidence paths:")
    for name in EVIDENCE_FILES:
        lines.append("  " + str(output_dir / name))
    lines.append("next action: inspect the persisted evidence and the failed stage above, then rerun the one-run cell after fixing the root cause.")
    msg = "\n".join(lines)
    print("\n--- ACTIONABLE ERROR ---")
    print(msg)
    raise ScientificSmokeExecutionError(msg)

def _validate_continuous_precondition(output_dir):
    output_dir = Path(output_dir)
    evidence = _load_smoke_evidence(output_dir)
    cp = evidence.get("checkpoint.json") or {}
    if not isinstance(cp, dict):
        cp = {}
    records = evidence.get("run_records.jsonl") or []
    if not isinstance(records, list):
        records = []
    sync = evidence.get("remote_sync.json") or {}
    if not isinstance(sync, dict):
        sync = {}
    problems = []
    if len(cp.get("succeeded_run_ids", [])) != 1:
        problems.append("succeeded != 1 (got %s)" % len(cp.get("succeeded_run_ids", [])))
    if len(cp.get("failed_run_ids", [])) != 0:
        problems.append("failed != 0 (got %s)" % len(cp.get("failed_run_ids", [])))
    if len(cp.get("pending_run_ids", [])) != 8:
        problems.append("pending != 8 (got %s)" % len(cp.get("pending_run_ids", [])))
    if cp.get("source_commit") != SOURCE_COMMIT:
        problems.append("source_commit mismatch")
    if cp.get("deployed_build_id") != DEPLOYED_BUILD_ID:
        problems.append("deployed_build_id mismatch")
    if not records:
        problems.append("run_records.jsonl missing")
    else:
        latest = records[-1]
        if int(latest.get("total_workflow_model_calls", 0) or 0) <= 0:
            problems.append("latest model_calls <= 0")
        if not latest.get("migration_generation_passed"):
            problems.append("migration not passed")
        if not latest.get("baseline_validation_passed"):
            problems.append("baseline not passed")
        if not latest.get("scenario_evaluator_passed"):
            problems.append("evaluator not passed")
    if sync.get("last_sync") != "recovery_uploaded":
        problems.append("HF state != recovery_uploaded (got %s)" % sync.get("last_sync"))
    if problems:
        print("\n--- CONTINUOUS PRECONDITION NOT MET ---")
        for p in problems:
            print("  -", p)
        _display_smoke_dashboard(output_dir)
        raise ScientificSmokeExecutionError("Continuous execution blocked: " + "; ".join(problems))
    print("\nCONTINUOUS PRECONDITION MET - safe to launch.")

def _run_benchmark_live(exec_cmd, output_dir, tail_limit=200):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    console_path = output_dir / "kaggle_console.log"

    sub_env = os.environ.copy()
    if not sub_env.get("HF_TOKEN", "").strip():
        raise RuntimeError("HF_TOKEN was not propagated to subprocess environment")
    sub_env["PYTHONPATH"] = str(CODE_DIR / "src") + (
        os.pathsep + sub_env["PYTHONPATH"] if sub_env.get("PYTHONPATH") else ""
    )
    sub_env["PYTHONUNBUFFERED"] = "1"
    sub_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    sub_env["TOKENIZERS_PARALLELISM"] = "false"

    print("Running:", " ".join(str(x) for x in exec_cmd))
    print("\n--- Output ---")
    tail = []
    with console_path.open("a", encoding="utf-8") as console:
        proc = subprocess.Popen(
            exec_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=sub_env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            console.write(line)
            tail.append(line)
            if len(tail) > tail_limit:
                tail.pop(0)
    return_code = proc.wait()
    print(f"\nReturn code: {return_code}")
    if return_code != 0:
        _raise_actionable_smoke_error(output_dir)
    return tail

# Post-execution scientific guardrail: raise unless the last persisted run is
# a real Qwen success and the required HF sync completed.
def _verify_scientific_run() -> None:
    cp_path = OUTPUT_DIR / "checkpoint.json"
    if not cp_path.is_file():
        raise RuntimeError("Guardrail failed: checkpoint.json missing")
    cp = _json.loads(cp_path.read_text(encoding="utf-8"))
    records_path = OUTPUT_DIR / "run_records.jsonl"
    if not records_path.is_file():
        raise RuntimeError("Guardrail failed: run_records.jsonl missing")
    records = [
        _json.loads(line)
        for line in records_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not records:
        raise RuntimeError("Guardrail failed: no run records")
    latest = records[-1]
    checks = {
        "latest run status = succeeded": latest.get("status") == "succeeded",
        "model_calls > 0": latest.get("total_workflow_model_calls", 0) > 0,
        "model identity starts with qwen:": str(cp.get("model_identity", "")).startswith("qwen:"),
        "baseline validation passed": latest.get("baseline_validation_passed") is True,
        "migration generation passed": latest.get("migration_generation_passed") is True,
        "scenario evaluator passed": latest.get("scenario_evaluator_passed") is True,
    }
    sync_path = OUTPUT_DIR / "remote_sync.json"
    if not sync_path.is_file():
        raise RuntimeError("Guardrail failed: remote_sync.json missing")
    sync = _json.loads(sync_path.read_text(encoding="utf-8"))
    synced = sync.get("last_sync", "") in ("recovery_uploaded", "snapshot_uploaded", "final_uploaded")
    checks["HF sync successful"] = synced
    for label, ok in checks.items():
        if not ok:
            raise RuntimeError(f"Guardrail failed: {label}")
    print("GUARDRAIL: PASSED - latest run succeeded with real Qwen calls and HF sync")


In [ ]:
# R7C correction: install the exact pinned runtime lock and verify imports/versions.
import importlib
import importlib.metadata

LOCK_PATH = CODE_DIR / "requirements-smoke-kaggle.lock"
if not LOCK_PATH.is_file():
    raise FileNotFoundError(f"pinned runtime lock missing in code dataset: {LOCK_PATH}")

PYTHON_RUNTIME = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError(
        f"Unsupported Python runtime {PYTHON_RUNTIME}; expected Python 3.11 or 3.12"
    )
print(f"Python runtime: {PYTHON_RUNTIME} [OK]")

print("Installing exact pinned Smoke runtime lock ...")
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(LOCK_PATH)],
    text=True,
)
if install.returncode != 0:
    raise RuntimeError("pip install of requirements-smoke-kaggle.lock failed")

EXPECTED_RUNTIME = {
    "django": ("Django", "django", "5.2.16"),
    "djangorestframework": ("djangorestframework", "rest_framework", "3.17.1"),
    "pytest": ("pytest", "pytest", "8.4.2"),
    "pytest_django": ("pytest-django", "pytest_django", "4.12.0"),
    "accelerate": ("accelerate", "accelerate", "1.14.0"),
    "bitsandbytes": ("bitsandbytes", "bitsandbytes", "0.49.2"),
}

print("\n--- PINNED RUNTIME VERSION TABLE ---")
mismatches = []
runtime_versions = {}
for key, (distribution, module_name, expected) in EXPECTED_RUNTIME.items():
    try:
        importlib.import_module(module_name)
        actual = importlib.metadata.version(distribution)
    except Exception as exc:
        actual = f"NOT_INSTALLED ({exc.__class__.__name__})"
    runtime_versions[key] = actual
    ok = "OK" if actual == expected else "MISMATCH"
    print(f"  {key:20s} expected={expected} actual={actual} [{ok}]")
    if actual != expected:
        mismatches.append(f"{key}={actual} (expected {expected})")
if mismatches:
    raise RuntimeError("pinned runtime version mismatch: " + "; ".join(mismatches))

env_meta = {
    "schema": "kaggle_runtime_environment.v1",
    "source_commit": SOURCE_COMMIT,
    "python_version": PYTHON_RUNTIME,
    "runtime_versions": runtime_versions,
}
RUNTIME_META_DIR = OUTPUT_DIR.parent / "environment"
RUNTIME_META_DIR.mkdir(parents=True, exist_ok=True)
(RUNTIME_META_DIR / "runtime_environment.json").write_text(
    _json.dumps(env_meta, indent=2, sort_keys=True), encoding="utf-8")
print("\nRUNTIME INSTALL + VERIFICATION: PASSED")


## R7C-REAL-RUN-ROOT-CLOSURE — Kaggle smoke preflight gate

This cell runs `--kaggle-preflight-only` before any experiment is created. It validates:

1. **Pinned runtime** — exact installed versions of Django, DRF, pytest, pytest-django, accelerate, bitsandbytes, torch, transformers
2. **Baseline Todo workspace** — `manage.py check` + `makemigrations todo --check --dry-run`
3. **int8 Qwen load** — `load_in_8bit=True` + `device_map="auto"` with `expandable_segments` allocator
4. **Deterministic 64-token probe** — seeded `torch.manual_seed(0)`
5. **VRAM headroom** — >= 2.0 GiB free after the probe

On failure it raises before any experiment, RunRecord, workspace result, or HF state is created. The machine-readable result is written to `kaggle_smoke_preflight.v1.json`.


In [ ]:
# R7C-REAL-RUN-ROOT-CLOSURE: preflight gate before any experiment is created.
PREFLIGHT_DIR = OUTPUT_DIR.parent / "preflight"
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

preflight_cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--kaggle-preflight-only",
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(PREFLIGHT_DIR),
]
preflight = subprocess.run(preflight_cmd, capture_output=True, text=True, timeout=1800)
print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(
        "KAGGLE SMOKE PREFLIGHT FAILED - no experiment will be started. "
        "Inspect the preflight table above and fix the root cause."
    )
print("KAGGLE SMOKE PREFLIGHT: PASSED")


## Engineering cross-session resume validation — one run per invocation

This cell runs the benchmark with `--auto-resume-hf` and `--max-runs 1`, which:

1. **Discovers** compatible experiments on Hugging Face under the canonical prefix:
   `experiments/{profile}/{protocol_version}/{source_commit}/`

2. **Downloads** each candidate's `checkpoint.json` and `run_records.jsonl`

3. **Validates** compatibility using explicit checkpoint identity fields:
   - `scenario_ids`, `strategy_names`, `planned_run_ids` (authoritative)
   - Profile, protocol version, source commit, config hash, model identity

4. **Selects** the action:
   - **RESUME** — skips completed runs, continues from the next pending arm
   - **ALREADY_COMPLETE** — all planned runs finished; exits cleanly
   - **START_NEW** — no compatible experiment found; creates a new experiment

5. **Logs** every candidate and rejection reason at INFO level with full diagnostic detail

**Completed runs are skipped.** Each re-execution cell advances the experiment:
- Session 1: `Terminal: 0/9 → 1/9`
- Session 2: `Terminal: 1/9 → 2/9`
- Session 3: `Terminal: 2/9 → 3/9`
- Session 4: `Terminal: 3/9 → 4/9`
- Session 5: `Terminal: 4/9 → 5/9`
- Session 6: `Terminal: 5/9 → 6/9`
- Session 7: `Terminal: 6/9 → 7/9`
- Session 8: `Terminal: 7/9 → 8/9`
- Session 9: `Terminal: 8/9 → 9/9`
- ... and so on until `Terminal: 9/9`.

**If `START_NEW` appears despite a compatible incomplete experiment existing,**
stop execution and investigate the rejection reasons logged at INFO level.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if not hf_token or not hf_token.strip():
    raise RuntimeError("HF_TOKEN Kaggle secret is missing or blank")

os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN: retrieved and set in environment")

In [ ]:
import subprocess

exec_cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--max-runs", "1",
    "--max-attempts", "3",
    "--protocol-version", "1.0",
    "--max-completion-tokens-per-call", "1024",
    "--max-total-workflow-tokens", "0",
    "--timeout", "300",
    "--hf-sync",
    "--auto-resume-hf",
    "--hf-repo-id", HF_RESULTS_REPO_ID,
    "--source-commit", SOURCE_COMMIT,
    "--deployed-build-id", DEPLOYED_BUILD_ID,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
tail = _run_benchmark_live(exec_cmd, OUTPUT_DIR)
_verify_scientific_run()
_display_smoke_dashboard(OUTPUT_DIR)
print("LAST_CONSOLE_LINES")
print("\n".join(tail[-10:]))


In [ ]:
import json
from pathlib import Path

run_dir = OUTPUT_DIR
print(f"Output directory: {run_dir}")

# Experiment ID
exp_id_path = run_dir / "experiment_id.txt"
if exp_id_path.exists():
    exp_id = exp_id_path.read_text().strip()
    print(f"Experiment ID: {exp_id}")
else:
    print("Experiment ID: (not found)")

# Checkpoint (authoritative source of truth)
cp_path = run_dir / "checkpoint.json"
if cp_path.exists():
    cp = json.loads(cp_path.read_text())
    total = cp.get("total_planned", 0)
    completed = len(cp.get("completed_run_ids", []))
    failed = len(cp.get("failed_run_ids", []))
    pending = len(cp.get("pending_run_ids", []))
    scenario_ids = cp.get("scenario_ids", [])
    strategy_names = cp.get("strategy_names", [])
    print(f"\nCheckpoint:")
    print(f"  Total:              {total}")
    print(f"  Completed:          {completed}")
    print(f"  Failed:             {failed}")
    print(f"  Pending:            {pending}")
    status = cp.get("completion_status", "unknown")
    print(f"  Completion status:  {status}")
    print(f"  Scenario IDs:       {scenario_ids}")
    print(f"  Strategy names:     {strategy_names}")
    ident = cp.get("identity_source", "unknown")
    print(f"  Identity source:    {ident}")
else:
    print("Checkpoint: (not found)")

# HF sync state
sync_path = run_dir / "remote_sync.json"
if sync_path.exists():
    sync = json.loads(sync_path.read_text())
    print(f"\nHF sync:")
    last = sync.get("last_sync", "unknown")
    print(f"  Last sync status:  {last}")
    timestamp = sync.get("timestamp", "unknown")
    print(f"  Timestamp:         {timestamp}")
    remote = sync.get("remote_path", "unknown")
    print(f"  Remote path:       {remote}")
    details = sync.get("details", "unknown")
    print(f"  Details:           {details}")
else:
    print("HF sync: (not found)")

## Continuous clean smoke — run remaining plan until 9/9 or interruption

This cell runs the benchmark **without** `--max-runs`, so it continues until all 9 runs finish or the session is interrupted.

Do **not** run this cell automatically. It is NOT safe to run until the
one-run cell above has produced at least 1/9 succeeded with the guardrail
passing (real Qwen model calls, validations, evaluator, and HF sync).


In [ ]:
import subprocess

_validate_continuous_precondition(OUTPUT_DIR)

exec_cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--backend", "kaggle-qwen",
    "--profile", "scientific-smoke-v2",
    "--max-attempts", "3",
    "--protocol-version", "1.0",
    "--max-completion-tokens-per-call", "1024",
    "--max-total-workflow-tokens", "0",
    "--timeout", "300",
    "--hf-sync",
    "--auto-resume-hf",
    "--hf-repo-id", HF_RESULTS_REPO_ID,
    "--source-commit", SOURCE_COMMIT,
    "--deployed-build-id", DEPLOYED_BUILD_ID,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
tail = _run_benchmark_live(exec_cmd, OUTPUT_DIR)
_verify_scientific_run()
_display_smoke_dashboard(OUTPUT_DIR)
print("LAST_CONSOLE_LINES")
print("\n".join(tail[-10:]))


## Notes

- **Scientific Smoke V2**: 1 repo (todo) x 3 scenarios x 3 arms x 1 run = 9 runs, non-publication.
- All outputs go to `/kaggle/working/runs/scientific_smoke_v2/`.
- Internet is required for Hugging Face result synchronization.
- `HF_TOKEN` is required and read from Kaggle Secrets.
- Qwen model loading remains offline from the attached Kaggle Model.
- To start a new experiment intentionally, add `--new-experiment` to the command in the execution cell.
